# EECS 182, Fall 2026 - HW02: ReLU with Different Optimizers

Use **Run all** to execute every active cell from top to bottom. The supplied checkpoint files let you make the required plots without training hundreds of networks. Leave the optional from-scratch training cell commented out.

Run in Google Colab, or open this notebook from its supplied `code` folder after installing the packages in `requirements.txt`. Answer the three written questions in the homework. No optimizer implementation is required in this exercise.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Use the supplied HW02 copy locally, or obtain it from the Fall 2026 repository.
relative = Path('hw02/q_relu_optim/code')
candidates = [Path.cwd(), Path.cwd() / relative,
              Path.cwd() / 'homework/hw02/q_relu_optim/code']
code_dir = next((p for p in candidates if (p / 'helpers.py').is_file()
                 and (p / 'ckpts/tensors.pth').is_file()), None)
if code_dir is None:
    repo = Path.cwd() / 'cs182fa26_public'
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/Berkeley-CS182/cs182fa26_public.git',
                        str(repo)], check=True)
    code_dir = repo / relative
    if not (code_dir / 'helpers.py').is_file():
        raise FileNotFoundError('Use the Fall 2026 HW02 code folder, including helpers and ckpts.')
os.chdir(code_dir)
sys.path.insert(0, str(Path.cwd()))
print('Using:', Path.cwd())


In [ ]:
# %load_ext autoreload
# %autoreload 2

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import copy
import time
import os
import sys
from ipywidgets import fixed, interactive, widgets 
from tqdm import tqdm

from helpers import *

%matplotlib inline

# Generate Training and Test Data

We are using piecewise linear function. Our training data has added noise $y = f(x) + \epsilon,\, \epsilon \sim \mathcal{N}(0, \sigma^2)$. The test data is noise free.

_After you have completed the exercise, you may wish to adjust the number of training samples and noise variance to see how gradient descent behaves under the new conditions._

In [ ]:
f_type = 'piecewise_linear'

def f_true(X, f_type):
    if f_type == 'sin(20x)':
        return np.sin(20 * X[:,0])
    else:
        TenX = 10 * X[:,0]
        _ = 12345
        return (TenX - np.floor(TenX)) * np.sin(_ * np.ceil(TenX)) - (TenX - np.ceil(TenX)) * np.sin(_ * np.floor(TenX)) 
    
n_features = 1
n_samples = 200
sigma = 0.01
rng = np.random.RandomState(1)

# Generate train data
X = np.sort(rng.rand(n_samples, n_features), axis=0)
y = f_true(X, f_type) + rng.randn(n_samples) * sigma

# Generate NOISELESS test data
X_test = np.concatenate([X.copy(), np.expand_dims(np.linspace(0., 1., 1000), axis=1)])
X_test = np.sort(X_test, axis=0)
y_test = f_true(X_test, f_type)

# Save checkpoint files
DIR_SGD = os.path.join(os.getcwd(), 'ckpts/sgd')
DIR_SGDM = os.path.join(os.getcwd(), 'ckpts/sgd_momentum')
DIR_ADAM = os.path.join(os.getcwd(), 'ckpts/adam')
os.makedirs(DIR_SGD, exist_ok=True)
os.makedirs(DIR_SGDM, exist_ok=True)
os.makedirs(DIR_ADAM, exist_ok=True)

def get_ckpt_dir(optim: str):
    if optim == 'sgd':
        return DIR_SGD
    elif optim == 'sgd_momentum':
        return DIR_SGDM
    elif optim == 'adam':
        return DIR_ADAM
    else:
        raise NotImplementedError


# Define the Neural Networks

We use one-hidden-layer ReLU networks, with widths 10, 20, and 40. For each width, compare SGD, SGD with momentum, and Adam at learning rate 0.02. All layers are trainable.

The first-layer biases place the initial ReLU elbows in [0, 1]. The optional fresh-training setup uses matching initial weights and elbows across optimizers for each seed. The supplied historical runs have random initializations; use their aggregate curves rather than treating two individual runs as a controlled comparison.


In [ ]:
# The default workflow uses the supplied checkpoints. Keep the optional training cell commented.
USE_PRETRAINED = True
nets_by_size = {}
nn_widths = [10, 20, 40]
nn_optimizer = ['sgd', 'sgd_momentum', 'adam']
nn_seeds = [442, 370, 378, 892, 836, 209, 327, 316, 216, 308,
            748, 934, 558, 546, 266, 808, 884, 818, 277, 979,
            766, 274, 479, 325, 431, 971, 689, 871, 272, 704]

def setup_networks(widths, optimizers, seed):
    torch.manual_seed(seed)
    rng = np.random.RandomState(seed)
    nets_by_size[seed] = {}
    for width in widths:
        net = nn.Sequential(nn.Linear(1, width), nn.ReLU(), nn.Linear(width, 1))
        # Use identical weights AND elbows for each optimizer in a fresh comparison.
        elbows = np.sort(rng.rand(width))
        with torch.no_grad():
            net[0].bias.copy_(to_torch(-elbows * to_numpy(net[0].weight).ravel()))
        nets_by_size[seed][width] = {}
        for name in optimizers:
            network = copy.deepcopy(net)
            if name == 'sgd':
                opt = torch.optim.SGD(network.parameters(), lr=0.02)
            elif name == 'sgd_momentum':
                opt = torch.optim.SGD(network.parameters(), lr=0.02, momentum=0.9)
            elif name == 'adam':
                opt = torch.optim.Adam(network.parameters(), lr=0.02)
            else:
                raise ValueError(name)
            nets_by_size[seed][width][name] = {'net': network, 'opt_all': opt,
                                              'optim': name, 'seed': seed}

for seed in nn_seeds:
    setup_networks(nn_widths, nn_optimizer, seed)


# Optional: train from scratch

The required workflow uses the supplied checkpoints. To explore fresh training, uncomment the following cell; it trains 270 networks for 30,000 steps each and can take substantial time. The final assignment sets `USE_PRETRAINED = False` so later cells use your new results. Rerunning the model-definition cell resets that choice and the models.


In [ ]:
# n_steps = 30000
# save_every = 3000 #1000
# t0 = time.time()

# def train_all_seeds(widths, optims, seeds):
  
#     for w in widths:
#       for i, optim in enumerate(optims):

#         print("-"*40)
#         print("Width", w, "Optimizer", optim)
#         list_of_history = []

#         print(f"training with {len(seeds)} seeds...")
#         for seed in tqdm(seeds):
#           net = nets_by_size[seed][w][optim]['net']
#           opt_all = nets_by_size[seed][w][optim]['opt_all']

#           save_dir = f'{get_ckpt_dir(optim)}/width{w}/seed{seed}/'
#           os.makedirs(save_dir, exist_ok=True)

#           history_all = train_network(X, y, X_test, y_test, 
#                                   net, optim=opt_all, 
#                                   n_steps=n_steps, save_every=save_every, 
#                                   verbose=False, optimizer=optim, seed=seed,
#                                   ckpt_dir=save_dir)
#           nets_by_size[seed][w][optim]['hist_all'] = history_all
#           list_of_history.append(history_all)  
      
# train_all_seeds(widths=nn_widths, optims=nn_optimizer, seeds=nn_seeds)
  
# t1 = time.time()
# print("-"*40)
# print("Trained all layers in %.1f minutes" % ((t1 - t0) / 60))

# # Compile the tensors into a dictionary
# data_dict = {
#     'X': X,
#     'y': y,
#     'X_test': X_test,
#     'y_test': y_test
# }

# torch.save(data_dict, "ckpts/tensors.pth")
# torch.save(nets_by_size, "ckpts/nets_by_size.pth")
# USE_PRETRAINED = False


The following cell loads all supplied histories and the matching final model states unless you ran the optional training cell.


In [ ]:
if USE_PRETRAINED:
    # These are the course-provided checkpoint files, copied into the fa26 homework.
    data = torch.load('ckpts/tensors.pth', map_location='cpu', weights_only=False)
    X, y, X_test, y_test = (data[key] for key in ['X', 'y', 'X_test', 'y_test'])
    for seed in nn_seeds:
        for width in nn_widths:
            for name in nn_optimizer:
                path = Path(get_ckpt_dir(name)) / f'width{width}/seed{seed}/ckpt_and_history.pt'
                history = torch.load(path, map_location='cpu', weights_only=False)
                entry = nets_by_size[seed][width][name]
                entry['hist_all'] = history
                entry['net'].load_state_dict(history[max(history)]['state'])

# All later plots use the same loaded or freshly trained histories and final states.
def histories_for(width, optimizer):
    return [nets_by_size[seed][width][optimizer]['hist_all'] for seed in nn_seeds]


## Plot Training Losses

Dots show the median across the 30 random seeds; error bars span the 25th to 75th percentiles. Each saved training loss is the MSE on that step's minibatch. The test curves below use the full noise-free test set.


In [ ]:
for width in nn_widths:
    fig, ax = plt.subplots(figsize=(12, 8))
    for i, name in enumerate(nn_optimizer):
        plot_with_error_bar(histories_for(width, name), optim=name,
                            plot_train=True, idx=i, ax=ax)
    ax.set_title(f'Width {width}; median and interquartile range across seeds')
    plt.show()


## Plot Test Losses

In [ ]:
for width in nn_widths:
    fig, ax = plt.subplots(figsize=(12, 8))
    for i, name in enumerate(nn_optimizer):
        plot_with_error_bar(histories_for(width, name), optim=name,
                            plot_test=True, idx=i, ax=ax)
    ax.set_title(f'Width {width}; median and interquartile range across seeds')
    plt.show()


## Visualize ReLU elbow positions

In [ ]:
SAMPLE = nn_seeds[0]
for w in nn_widths:
    for optim in nn_optimizer:
        fig, ax = plt.subplots(figsize=(12, 8))
        net = nets_by_size[SAMPLE][w][optim]['net']
        plot_update(X, y, X_test, y_test, net, optim=optim, ax=ax)
        ax.set_title(f"width {w}, {optim}")
        plt.show()

## TODO: Bug Hunt - Accumulating Gradients

The training loop below is shown for inspection only. It is a Markdown code block, not an executable cell, so **Run all will not execute the bug**.

```python
model.train()
optimizer.zero_grad()

for inputs, labels in training_loader:
    predictions = model(inputs)
    loss = loss_fn(predictions, labels)
    loss.backward()
    optimizer.step()
```

**TODO:**
1. Identify the bug and explain why it can make training unstable.
2. If the first two batch gradients are $g_1$ and $g_2$, what is stored in each parameter's `.grad` field after the second `loss.backward()` call?
3. Does `optimizer.step()` clear `.grad`?
4. Write or state the corrected per-batch operation order. You do not need to edit or execute this snippet.


## Questions

Questions to consider while exploring the training process for all layer weights with different optimizers:
- **How does the hidden layer width and different optimizers impact the learned function and test error?**

- **What happens to the elbow locations using different optimizers during training?**
<!--
- **Check out the `helpers.py` file. Are the circle dots on the test loss graphs above mean or median? Why not plot the other one?**

-  **How are error bars for the test losses computed? Are the upper and lower marks maximum/minimum, standard deviation or something else? Does it make sense to plot standard deviation here?**
-->

In [ ]:
# Create a single figure with a 3x3 grid of subplots
fig, axs = plt.subplots(3, 3, figsize=(15, 15))

# Flatten the axs array for easy indexing
axs = axs.ravel()

# Iterate through the widths and optimizers and plot in subplots
for i, w in enumerate(nn_widths):
    for j, optim in enumerate(nn_optimizer):
        ax = axs[i * 3 + j]
        net = nets_by_size[SAMPLE][w][optim]['net']
        plot_update(X, y, X_test, y_test, net, optim=optim, ax=ax)
        ax.set_title(f"width {w}, {optim}")

# Adjust subplot layout
plt.tight_layout()
plt.show()
